# Evidence Ledger
### Roadrunner propulsion reverse-engineering study · notebook 1 of 15

**Purpose.** Everything this study concludes rests on what is publicly known about
the vehicle, and very little is. This notebook lays that evidence out before any
calculation is done, so that every later number can be traced to either a cited
source or a stated assumption (ADR-0005).

> *What do we actually know, how firmly, and where do the sources disagree?*

The manufacturer has published **no figure** for speed, range, endurance, altitude,
mass, thrust or dimensions. What exists is a handful of on-record qualitative
statements, a few secondary reports, and marketing comparatives.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import load_claims, load_mission, reports
from roadrunner_re.evidence import write_evidence_handoff
from roadrunner_re.notebook import nb_setup

paths = nb_setup()

---
## 1 · The ledger, by tier

Each entry lives in `config/evidence/*.yaml` with its source URL and access date.
To change one, edit the YAML and re-run — **do not hard-code numbers in this notebook.**

| Tier | Meaning | Use |
|---|---|---|
| A | Stated by the manufacturer on the record | Input |
| B | Secondary report, or our own measurement | Input, with wide bounds |
| C | Inferred by analogy or judgement | Input, with wide bounds |
| X | Marketing comparative with no stated baseline | **Never an input**; tested at the end |

In [ ]:
claims = load_claims()
print(reports.evidence_table(claims))

---
## 2 · What is quantified

Only three quantities have any number attached, and all three come from secondary
sources. The *spread* column is the width of the bounds relative to the nominal
value; it is the honest size of our ignorance and it propagates into every result
(ADR-0020).

In [ ]:
print(reports.quantified_table(claims))

The overall length is the **single anchor** for all geometry measured from imagery
(ADR-0007): the diameters that bound the engines scale directly with it, so its
±spread is inherited by everything downstream.

---
## 3 · Conflicts and cautions

Where sources disagree, or a source is weak, the ledger says so rather than picking
a winner.

In [ ]:
print(reports.cautions(claims))

---
## 4 · Which claims the sample mission leans on

The mission profile (`config/missions/sample.yaml`, ADR-0018) takes its *shape* from
these claims. Its *numbers* are assumptions: no source gives any of them.

In [ ]:
mission = load_mission(claims=claims)
for seg in mission.segments:
    print(f"{seg.id:12s} {seg.type:17s} cites: {', '.join(seg.basis.evidence) or '(assumption only)'}")
print(f"\n{len(mission.uncertain_inputs())} mission values are uncertain inputs to the uncertainty analysis.")

---
## 5 · Handoff

In [ ]:
handoff = write_evidence_handoff(claims, paths.out / "evidence.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- No performance or size figure comes from the manufacturer. Every quantified input
  is a secondary report with wide bounds.
- The reported mass (85 kg) and top speed (700 km/h) come from one uncorroborated
  source; the speed sits at the low end of what "high subsonic" would normally mean.
- Sources disagree on overall length (1.5 m against 1.8 m) and on the number of
  landing legs. Imagery should settle the second; nothing public settles the first.
- Two claims are marketing comparatives and are held back as hypotheses.